# DL vertex finding network training

This notebook is designed to take the input and truth images generated by the <code>make_images.ipynb</code> notebook and train networks for vertex finding. This notebook generates models for each of the U, V and W views for each of the required passes.
    
Most of the cells below will not need any editing, but towards the bottom of the notebook you will find some additional markdown that describes what you may need to edit (essentially just some file locations).

In [ ]:
# Automatically reload external libraries that change
%reload_ext autoreload
%autoreload 2

# If a matplotlib plot command is issued, display the results in the notebook
%matplotlib inline

In [ ]:
from network import set_seed, get_class_weights
from network import load_model_only, load_model, save_model, create_model
from network import accuracy

from data import SegmentationDataset, SegmentationBunch


import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import torch

from __future__ import annotations
from cycler import cycler

PlotStyle = {
    'axes.ymargin': 0.1,
    'legend.frameon': False,
    'xaxis.labellocation': 'right',
    'yaxis.labellocation': 'top',
    
    'axes.formatter.limits': (-2, 3),

    # 2. ax.ticklabel_format(useMathText=True)
    'axes.formatter.use_mathtext': True, 

    # 3. ax.minorticks_on()
    'xtick.minor.visible': True, 
    'ytick.minor.visible': True, 

    'xtick.major.size': 6, 
    'ytick.major.size': 6, 

    'xtick.labelsize': 14, 
    'ytick.labelsize': 14, 

    'xtick.direction': 'in', 
    'ytick.direction': 'in', 

    'xtick.top': True, 
    'ytick.right': True, 

    # 2. ax.tick_params(which='minor', length=3, direction='in', right=True, top=True)
    'xtick.minor.size': 3, 
    'ytick.minor.size': 3, 

    # Note: Direction, top, and right settings automatically apply to minor ticks 
    # when set globally, but you can explicitly ensure they mirror major ticks.
    'xtick.minor.top': True, 
    'ytick.minor.right': True, 

    'axes.xmargin': 0.0, 

    'legend.title_fontsize': 16, 
    'legend.fontsize': 14, 
    'axes.labelsize': 17, 
    'axes.titlesize': 16,

    'legend.handleheight': 1, 
    'legend.handlelength': 1.2, 

    'savefig.dpi': 300, 

    'axes.prop_cycle': (
        cycler('color', ["#0d49fb", "#e6091c", "#26eb47", "#8936df", "#fec32d", "#25d7fd"]) + 
        cycler('ls', ['-', '--', '-.', ':', '-', '--'])
    ), 
    'savefig.transparent': True,
    'savefig.bbox': 'tight',
}

plt.style.use(PlotStyle)

# Run network training here

The key parameters that will need editing are the view and the pass to be trained. Each view/pass combination has its own network. Views are specified using the standard U, V, W nomenclature (and is consistent with the file naming conventions from previous steps), while the pass is either pass 1 or pass 2.

The respective variables can be set in the cell below via <code>view</code> and <code>vertex_pass</code>.

If you edited the <code>thresholds</code> variable at the <code>make_images</code> stage, you may need to update the <code>NUM_CLASSES</code> variable to reflect any change in the number of thresholds. Note that this value should be equal to the length of the <code>thresholds</code> variable, despite this variable specifying bin edges, because one extra class is required to represent the null case where a pixel has no hits in it.

<code>batch_size</code> can, of course, be varied according to the available resources of your GPU, but as a semantic segmentation network you'll need a lot of memory on your GPU to increase this beyond the current default of 32.

The <code>image_path</code> variable should contain the path to the images generated by the <code>make_images</code> notebook (i.e. <code>global_path</code>), and will expect to find the <code>Hits</code> and <code>Truth</code> folders within that path).

> Note that the code has been updated to allow for sample balancing: taking as input the `balance_map` dictionary, it can, for each sample class, consider a different fraction.
> The dict. is required to follow this schema
> ```python
> {
>   'class_1': {'dir': 'path_1', 'fraction': 0.75},
>   'class_2': {'dir': 'path_2', 'fraction': 0.25}
> }
> ```

Note that the cells below will count the class representation in the training set, determine how to weight them and print this out. It's worth taking a look at this output to ensure all classes are represented, as training will fail if they are not.

You will want to set the number of epochs, <code>n_epochs</code> to train for. This is not easy to determine a priori, but 20 is a reasonable starting point (plots of the loss function and accuracy are produced to help you determine when the network has effectively trained).

<code>model_name</code> acts as a prefix for saving the model. The state of the model is saved after every epoch, with a suffix indicating the epoch number.

Once you are happy with the variable values you can run all of the cells in this section in order (having run all of the cells above), with the final cell in this section actually performing the training.

# First training: perfectly balanced 50-50/50-50 samples

In [ ]:
# This line is important for GPU running, otherwise some weights end up on the CPU
torch.set_default_tensor_type(torch.cuda.FloatTensor)

view = "W"
vertex_pass = 1
the_seed = 42
gpu = torch.device('cuda:0')
batch_size=150
NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19
# image_path = f"Accel/Pass{vertex_pass}/Images{view}"
# image_path = '/exp/icarus/data/users/msotgia/vertexStudies/forTraining/ICARUS_DlVertex'
image_path = '/home/msotgia/vertexOnEaf/ICARUS_DlVertex_HDF5' # For processing on EAF w/o overhead

balance_map = {
    'numi_numu': {'dir': f'NuMI/numu/Pass{vertex_pass}/Images{view}', 'fraction': 0.25},
    'numi_nue':  {'dir': f'NuMI/nue/Pass{vertex_pass}/Images{view}',  'fraction': 0.25},
    'bnb_numu':  {'dir': f'BNB/numu/Pass{vertex_pass}/Images{view}',  'fraction': 0.25},
    'bnb_nue':   {'dir': f'BNB/nue/Pass{vertex_pass}/Images{view}',   'fraction': 0.25}
}

n_epochs = 20
model_name = "icarus_pass1_fully_balanced_beam_flavour"

for subdir in ["models", "stats", "images"]:
    dir = f"outputs/{subdir}/pass{vertex_pass}/{view}"
    if not os.path.exists(dir):
        os.makedirs(dir)

In [ ]:
# main.py

#from data import *
#from network import *

set_seed(the_seed)
bunch = SegmentationBunch(image_path, balance_map, batch_size=batch_size, valid_pct = 0.25, device=gpu)
train_stats = bunch.count_classes(NUM_CLASSES)
weights = get_class_weights(train_stats)

`weights` before the changes were

```python
[np.float64(8.623750719476234e-08),
 np.float64(0.007172987689688895),
 np.float64(0.0007584045062301771),
 np.float64(0.00037038372234998225),
 np.float64(0.00024288647877137123),
 np.float64(0.000198978541093642),
 np.float64(0.00019057576343509776),
 np.float64(0.0001858506259661026),
 np.float64(0.0003745255916270518),
 np.float64(0.000288019790190076),
 np.float64(0.00048425659795097424),
 np.float64(0.0007499850008340051),
 np.float64(0.001114525094378949),
 np.float64(0.0016114242089710934),
 np.float64(0.0016443771616277915),
 np.float64(0.002822851947461105),
 np.float64(0.004856441516708472),
 np.float64(0.011101357805652648),
 np.float64(0.08215330262912612),
 np.float64(0.8836787790904292)]
 ```

 Below is after 

In [ ]:
weights

In [ ]:
np.savez(f'outputs/stats/pass{vertex_pass}/{view}/weights_{model_name}.npz', weights)

In [ ]:
train_losses = torch.zeros(n_epochs * len(bunch.train_dl), device=gpu)
val_losses = torch.zeros(n_epochs, device=gpu)
batch_losses = torch.zeros(len(bunch.valid_dl), device=gpu)

train_accs = torch.zeros(n_epochs * len(bunch.train_dl), device=gpu)
val_accs = torch.zeros(n_epochs, device=gpu)
batch_accs = torch.zeros(len(bunch.valid_dl), device=gpu)

In [ ]:
# Standard model creation
model, loss_fn, optim = create_model(NUM_CLASSES, weights, gpu)

i = 0
start = 0
finish = n_epochs

In [ ]:
from tqdm.notebook import tqdm
set_seed(the_seed)
for e in tqdm(range(start, finish), desc='Training'):
    model = model.train()
    n_batches = len(bunch.train_dl)
    for b, batch in enumerate(tqdm(bunch.train_dl, desc=f'For epoch {e}, training over batches')):
        x, y = batch
        pred = model.forward(x)
        loss = loss_fn(pred, y)

        train_losses[i] = loss.item()
        train_accs[i] = accuracy(pred, y, nearby=False)

        loss.backward()
        optim.step()
        #scheduler.step()
        optim.zero_grad()
        i += 1
        if b == (n_batches - 1):
            save_model(model, x, f"outputs/models/pass{vertex_pass}/{view}/{model_name}_{e}")

    # Validate
    model = model.eval()
    with torch.no_grad():
        for b, batch in enumerate(tqdm(bunch.valid_dl, desc=f'For epoch {e}, validating over batches')):
            x, y = batch
            pred = model.forward(x)
            loss = loss_fn(pred, y)
            
            batch_losses[b] = loss.item()
            batch_accs[b] = accuracy(pred, y, nearby=False)
        val_losses[e] = torch.mean(batch_losses)
        val_accs[e] = torch.mean(batch_accs)

    np.savez(f'outputs/stats/pass{vertex_pass}/{view}/losses_{model_name}_{e}.npz', 
             train_losses.cpu(), val_losses.cpu(), batch_losses.cpu(), train_accs.cpu(), val_accs.cpu(), batch_accs.cpu())

    

# Assess network performance

Having trained a network, you can look at its performance - this should be run immediately after the network has finished training. The cells below should not require any editing.

The first cell runs over a single batch from the validation set and produuces images allowing you to compare the truth (left image) to the network classification (right image), though it is worth noting that the <code>show_batch</code> function produces all images in the same folder and does not uniquely identify the view or pass, so if you want to retain them, you'll want to move them between runs.

The next three cells produce plots showing the evolution of the network across epochs. Ideally you want to see a plateauing of the loss and accuracy to establish a well trained model, with no evidence that the training and validation performance are diverging (you can always select a model from an earlier epoch before divergence if the network appears to be overfitting - or get more training samples if the network is not adequately trained).

The final cell in this section simply saves the evolution history of the network to allow easy plot regenertion if needed.

In [ ]:
set_seed(the_seed)
model = model.eval()
with torch.no_grad():
    for b, batch in enumerate(bunch.valid_dl):
        x, y = batch
        pred = model.forward(x)
        show_batch(finish, b, x, pred, y, n=32, randomize=False)
        break

In [ ]:
VIEW = 'W'
OUTPUT_PATH = 'outputs'
VERTEX_PASS = 1
MODEL_NAME = 'icarus_fully_balanced_beam_flavour'

CACHE_PATH = f'{OUTPUT_PATH}/cache/pass{VERTEX_PASS}/{VIEW}'
MODEL_PATH = f'{OUTPUT_PATH}/models/pass{VERTEX_PASS}/{VIEW}'
IMAGES_PATH = f'{OUTPUT_PATH}/images/pass{VERTEX_PASS}/{VIEW}'
STATS_PATH = f'{OUTPUT_PATH}/stats/pass{VERTEX_PASS}/{VIEW}'

EPOCH = 44 # 0-indexed epoch of the final cache write
# RECORDED_EPOCHS = EPOCH + 1 
RECORDED_EPOCHS = 45

SAVED_EPOCH = 25

weights = np.asarray(np.load(f'{CACHE_PATH}/weights_{MODEL_NAME}.npz', allow_pickle=True)['arr_0'], 
                     dtype=np.float32)

cached_losses = np.load(f'{CACHE_PATH}/losses_{MODEL_NAME}_{EPOCH}.npz', allow_pickle=True)

train_losses = np.asarray(cached_losses['arr_0'], dtype=np.float32)
val_losses   = np.asarray(cached_losses['arr_1'], dtype=np.float32)
train_accs   = np.asarray(cached_losses['arr_3'], dtype=np.float32)
val_accs     = np.asarray(cached_losses['arr_4'], dtype=np.float32)

# Derive batches/epoch from the data instead of guessing (the guessed 150 above didn't
# match the real per-epoch batch count, which is what produced the sawtooth: reshaping
# with the wrong width misaligns every epoch after the first, mixing batches from the
# wrong epoch - and any zero-padding from a mismatched guess - into each averaged point).

if len(train_losses) % RECORDED_EPOCHS != 0:
    raise ValueError(
        f"train_losses length {len(train_losses)} isn't an exact multiple of "
        f"{RECORDED_EPOCHS} epochs - batches/epoch must have actually changed at some "
        f"point in this cache's history (different --batch-size, or a dataset resample "
        f"that changed len(bunch.train_dl)). A single uniform reshape can't represent "
        f"that; per-epoch boundaries would need to be tracked explicitly instead."
    )
n_train_batches = len(train_losses) // RECORDED_EPOCHS
print(f'Derived {n_train_batches} batches/epoch from the cached data '
      f'(rather than assuming 150)')

tl = train_losses.reshape(RECORDED_EPOCHS, n_train_batches).mean(axis=1)
tlerr = train_losses.reshape(RECORDED_EPOCHS, n_train_batches).std(axis=1)

ta = train_accs.reshape(RECORDED_EPOCHS, n_train_batches).mean(axis=1)
taerr = train_accs.reshape(RECORDED_EPOCHS, n_train_batches).std(axis=1)


fig, (ax, ax_ratio) = plt.subplots(figsize=(4, 4), nrows=2, sharex=True, height_ratios=[.7, .3])
fig.align_ylabels()
fig.tight_layout()
fig.subplots_adjust(wspace=0.01, hspace=0.075)

ax.set(
    # xlabel='Epochs', 
    ylabel='Loss', 
    xlim=(0, EPOCH)
)
ax.set_title(f'View {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')

# ---------------

ax.plot(tl, color='C0', lw=1.5)
ax.fill_between(np.arange(RECORDED_EPOCHS), tl-tlerr, tl+tlerr, color='C0', ec='none', alpha=0.5)
ax.plot(val_losses[:RECORDED_EPOCHS], color='C1', lw=1.5)
ax.add_artist(ax.legend(handles=[
    (mpl.lines.Line2D([0],[0], color='C0'),
     mpl.patches.Patch(fc='C0', ec='none', lw=0, alpha=0.25)), 
    mpl.lines.Line2D([0],[0], color='C1', ls='--')
], labels=['Training', 'Validation']))

ax.axvline(SAVED_EPOCH, c='k', ls=':', lw=1)

# ---------------
# Loss ratio (val / train), with training normalized to 1
tl_rel_err = tlerr / tl                      # relative training uncertainty -> band around 1
val_loss_ratio = val_losses[:RECORDED_EPOCHS] / tl

ax_ratio.set(
    xlabel='Epochs',
    ylabel='Ratio'
)

ax_ratio.plot(np.full(RECORDED_EPOCHS, 1.0), color='C0', lw=1.5)   # training -> flat at 1
ax_ratio.fill_between(np.arange(RECORDED_EPOCHS), 1 - tl_rel_err, 1 + tl_rel_err,
                       color='C0', ec='none', alpha=0.5)
ax_ratio.plot(val_loss_ratio, color='C1', lw=1.5, ls='--')          # validation varies

ax_ratio.axvline(SAVED_EPOCH, c='k', ls=':', lw=1)

fig.savefig(f'{STATS_PATH}/Loss_TrainingValidation_View{VIEW}.pdf')
fig.savefig(f'{STATS_PATH}/Loss_TrainingValidation_View{VIEW}.png')
plt.show()

# ---------------
# ---------------

# ---------------
# ---------------

fig, (ax, ax_ratio) = plt.subplots(figsize=(4, 4), nrows=2, sharex=True, height_ratios=[.7, .3])
fig.align_ylabels()
fig.tight_layout()
fig.subplots_adjust(wspace=0.01, hspace=0.075)

ax.set(
    # xlabel='Epochs', 
    ylabel='Accuracy', 
    xlim=(0, EPOCH)
)
ax.set_title(f'View {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')

# ---------------

ax.plot(ta, color='C0', lw=1.5)
ax.fill_between(np.arange(RECORDED_EPOCHS), ta-taerr, ta+taerr, color='C0', ec='none', alpha=0.5)
ax.plot(val_accs[:RECORDED_EPOCHS], color='C1', lw=1.5)
ax.add_artist(ax.legend(handles=[
    (mpl.lines.Line2D([0],[0], color='C0'),
     mpl.patches.Patch(fc='C0', ec='none', lw=0, alpha=0.25)), 
    mpl.lines.Line2D([0],[0], color='C1', ls='--')
], labels=[f'Training (WP = {ta[SAVED_EPOCH]:.1%})', f'Validation (WP = {val_accs[SAVED_EPOCH]:.1%})']))

ax.axvline(SAVED_EPOCH, c='k', ls=':', lw=1)

# ---------------
# Accuracy ratio (val / train), with training normalized to 1
ta_rel_err = taerr / ta
val_acc_ratio = val_accs[:RECORDED_EPOCHS] / ta

ax_ratio.set(
    xlabel='Epochs',
    ylabel='Ratio',
    xlim=(0, EPOCH)
)

ax_ratio.plot(np.full(RECORDED_EPOCHS, 1.0), color='C0', lw=1.5)
ax_ratio.fill_between(np.arange(RECORDED_EPOCHS), 1 - ta_rel_err, 1 + ta_rel_err,
                       color='C0', ec='none', alpha=0.5)
ax_ratio.plot(val_acc_ratio, color='C1', lw=1.5, ls='--')

ax_ratio.axvline(SAVED_EPOCH, c='k', ls=':', lw=1)

fig.savefig(f'{STATS_PATH}/Accs_TrainingValidation_View{VIEW}.pdf')
fig.savefig(f'{STATS_PATH}/Accs_TrainingValidation_View{VIEW}.png')
plt.show()


In [ ]:
with open(f'outputs/stats/pass{vertex_pass}/{view}/train_loss_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, tl)
with open(f'outputs/stats/pass{vertex_pass}/{view}/val_loss_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, vl)
with open(f'outputs/stats/pass{vertex_pass}/{view}/train_accs_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, ta)
with open(f'outputs/stats/pass{vertex_pass}/{view}/val_accs_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, va)

# Generating a TorchScript network

The network was trained on a GPU, but ultimately runs on a CPU in a C++ context. This means that the network must be converted to TorchScript format. This can be performed using the cell below and can be run at any time - it need not be run immediately after training the network, because it only requires access to a saved model state.

The only parameters requiring editing here are the location of the input file; which is the save model from the chosen training epoch (so some combination of the <code>moidel_name</code> and epoch with a <code>.pkl</code> extension), the <code>output_filename</code>, which should have a <code>.pt</code> extension, and also the number of classes <code>NUM_CLASSES</code>, which should, of course, match the previouslyt specified value.

The resultant <code>.pt</code> files are what will ultimately be loaded by Pandora for network inference.

In [ ]:
# This line is important for ensuring all tensors exist on the same device
# torch.set_default_tensor_type(torch.FloatTensor)

SAVING_EPOCH = 25

filename = f'{MODEL_PATH}/{MODEL_NAME}_{SAVING_EPOCH}.pkl'
output_filename = f'PandoraVertexNet_{MODEL_NAME}_{VERTEX_PASS}_{VIEW}.pt'
the_seed = 42
device = torch.device('cpu')
NUM_CLASSES = 20

set_seed(the_seed)

model = load_model_only(filename, NUM_CLASSES, device)

sm = torch.jit.script(model)
sm.save(output_filename)

# Confusion Matrices

### Computing epoch Confusion Matrix

In [ ]:
# This line is important for GPU running, otherwise some weights end up on the CPU
# torch.set_default_tensor_type(torch.cuda.FloatTensor)

BALANCE_MAP = {
    'NuMI_numu': {
        'dir': f'NuMI/numu/Pass{VERTEX_PASS}/Images{VIEW}', 'fraction': 0.25
    },
    'NuMI_nue': {
        'dir': f'NuMI/nue/Pass{VERTEX_PASS}/Images{VIEW}', 'fraction': 0.25
    },
    'BNB_numu': {
        'dir': f'BNB/numu/Pass{VERTEX_PASS}/Images{VIEW}', 'fraction': 0.25
    },
    'BNB_nue': {
        'dir': f'BNB/nue/Pass{VERTEX_PASS}/Images{VIEW}', 'fraction': 0.25
    }
}

NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19
BATCH_SIZE = 15
VALIDATION_PCT = 0.25
PATH = '/home/msotgia/vertexOnEaf/ICARUS_DlVertex_HDF5'

the_seed = 42
device = torch.device('cuda:0')

set_seed(the_seed)
bunch = SegmentationBunch(PATH, BALANCE_MAP, batch_size=BATCH_SIZE, valid_pct=VALIDATION_PCT, device=device)
filename = f'{MODEL_PATH}/{MODEL_NAME}_{SAVING_EPOCH}.pkl'

set_seed(the_seed)
model = load_model_only(filename, NUM_CLASSES, device)

In [ ]:
import scipy.stats as stats
from tqdm.auto import tqdm

binning = np.linspace(0, 20, 21, dtype=int)

model = model.to(device)
confusion = np.zeros((20,20))
for img, cls in tqdm(bunch.valid_dl, desc='Creating confusion matrix'):
    img = img.to(device)
    output = model(img)
    _, preds = torch.max(output, 1)
    
    cls_detached = cls.cpu().numpy().flatten()
    preds_detached = preds.cpu().numpy().flatten()
    
    H, *_ = stats.binned_statistic_2d(preds_detached, cls_detached, None,
                                      bins=[binning, binning], statistic='count')
    confusion += H

In [ ]:
temporary = confusion.copy()
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.set(
    xlabel='True class',
    ylabel='Fraction'
)

ax.step(list(np.arange(1, 20)), np.sum(temporary[1:], axis=1) / np.sum(temporary[1:]), where='mid')
ax.set_title(f'Epoch {SAVING_EPOCH}\nView {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')

fig.savefig(f'{STATS_PATH}/true_class_dist_{VERTEX_PASS}_{VIEW}.pdf')
fig.savefig(f'{STATS_PATH}/true_class_dist_{VERTEX_PASS}_{VIEW}.png')

In [ ]:
sums = np.sum(confusion, axis=1).repeat(20).reshape((20,20))
confusion /= sums

In [ ]:
print(f"--- Class Accuracy")
for t in range(confusion.shape[0]):
    print(f"{t:2}: {100*(confusion[t,t] / confusion[t].sum()):.1f}")
print()

In [ ]:
confusion[0,:] = 0
confusion[:,0] = 0

sums = np.sum(confusion, axis=1).repeat(20).reshape((20,20))
sums[0,:] = 1
confusion /= sums

print(f"--- Class Accuracy")
for t in range(confusion.shape[0]):
    print(f"{t:2}: {100*confusion[t,t]:.1f}")
print()

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.set(xlabel='True classes', ylabel='Network prediction')
ax.set_title(f'Epoch {SAVING_EPOCH}\nView {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')
m = ax.imshow(confusion, cmap='Blues', origin='lower')
fig.colorbar(m)
fig.savefig(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}.png')
fig.savefig(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}.pdf')
# save_figure(fig, f"outputs/stats/pass{vertex_pass}/{view}/confusion_{vertex_pass}_{view}")

ax.plot(np.arange(NUM_CLASSES), lw=1, c='w', ls='--')

np.savez_compressed(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_{SAVING_EPOCH}', confusion)

for t in range(20):
    for n in range(20):
        print(f"{confusion[t, n]:.2f}", end=" ")
    print()

## Confusion matrix comparison

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))

EPOCH_1, EPOCH_2 = 44, 43
VIEW='UV'
VERTEX_PASS=1


NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19

OUTPUT_PATH = 'outputs'

CACHE_PATH = f'{OUTPUT_PATH}/cache/pass{VERTEX_PASS}/{VIEW}'
MODEL_PATH = f'{OUTPUT_PATH}/models/pass{VERTEX_PASS}/{VIEW}'
IMAGES_PATH = f'{OUTPUT_PATH}/images/pass{VERTEX_PASS}/{VIEW}'
STATS_PATH = f'{OUTPUT_PATH}/stats/pass{VERTEX_PASS}/{VIEW}'

confusion_1 = np.load(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_{EPOCH_1}.npz')['arr_0']
confusion_2 = np.load(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_{EPOCH_2}.npz')['arr_0']

ax.set(xlabel='True classes', ylabel='Network prediction')

ax.set_title(f'Difference {EPOCH_1}/{EPOCH_2}\nView {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')
ratio = (confusion_1 - confusion_2)

# ax.set_title(f'Ratio {EPOCH_1}:{EPOCH_2}\nView {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')
# ratio = (confusion_1/confusion_2)

m = ax.imshow(ratio, cmap='seismic', origin='lower', norm=mpl.colors.CenteredNorm(vcenter=0))
fig.colorbar(m)

ax.plot(np.arange(NUM_CLASSES), lw=1, c='k', ls='--')

fig.savefig(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_comparing_{EPOCH_1}_{EPOCH_2}.png')
fig.savefig(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_comparing_{EPOCH_1}_{EPOCH_2}.pdf')


## Replot :)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))

EPOCH = 22
VIEW='UV'
VERTEX_PASS=1

confusion = np.load(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_{EPOCH}.npz')['arr_0']

ax.set(xlabel='True classes', ylabel='Network prediction')
ax.set_title(f'Epoch {EPOCH}\nView {VIEW} / Vertex pass {VERTEX_PASS}', color='gray', loc='right')
m = ax.imshow(confusion, cmap='Blues', origin='lower')
fig.colorbar(m)

ax.plot(np.arange(NUM_CLASSES), lw=1, c='w', ls='--')

fig.savefig(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_{EPOCH}.png')
fig.savefig(f'{STATS_PATH}/confusion_{VERTEX_PASS}_{VIEW}_{EPOCH}.pdf')

